In [10]:
from backtesting import Backtest, Strategy
import pandas as pd
import yfinance as yf

In [11]:
# descarga de datos
df_apple = yf.download('AAPL', period='60d', interval='2m')

# limpieza de datos
df_apple.columns = df_apple.columns.get_level_values(0)
df_apple.dropna(inplace=True)

# columna de retornos
df_apple['Return'] = df_apple['Close'].pct_change() * 100
df_apple.dropna(inplace=True)

[*********************100%***********************]  1 of 1 completed


In [13]:
# clase para la estrategia
class STD_Strategy(Strategy):
    # definimos tp y sl en porcentajes, tambien window
    sl_pct = 0.005
    tp_pct = 0.015
    window = 50
    num_std = 2
    
    # indicadores que usare
    def init(self):
        close = self.data.Close
        # promedio
        self.sma = self.I(lambda x: pd.Series(x).rolling(self.window).mean(), close)

        # desviacion estandar
        self.rolling_std = self.I(lambda x: pd.Series(x).rolling(self.window).std(), close)

    def next(self):
        price = self.data.Close[-1]
        media = self.sma[-1]
        desviacion = self.rolling_std[-1]

        if self.position:
            return
        
        if price < (media - self.num_std * desviacion):
            self.open_long(price)

        elif price > (media + self.num_std * desviacion):
            self.open_short(price)

# metodos de operacion
    def open_long(self, price):
        self.buy(
            sl=price * (1 - self.sl_pct),
            tp=price * (1 + self.tp_pct)
        )

    def open_short(self, price):
        self.sell(
            sl=price * (1 + self.sl_pct),
            tp=price * (1 - self.sl_pct)
        )

In [14]:
# configuracion del backtest
bt = Backtest(df_apple, STD_Strategy, cash=10000, commission=0.002)

# ejecucion
stats = bt.run()
print(stats)

Start                     2026-03-06 14:32...
End                       2026-04-24 19:58...
Duration                     49 days 05:26:00
Exposure Time [%]                    78.94197
Equity Final [$]                   7183.57345
Equity Peak [$]                   10093.12257
Commissions [$]                    2860.43328
Return [%]                          -28.16427
Buy & Hold Return [%]                  5.4339
Return (Ann.) [%]                   -90.92246
Volatility (Ann.) [%]                  2.1658
CAGR [%]                            -81.61017
Sharpe Ratio                        -41.98093
Sortino Ratio                        -3.36064
Calmar Ratio                         -3.11281
Alpha [%]                            -30.5972
Beta                                  0.44773
Max. Drawdown [%]                   -29.20916
Avg. Drawdown [%]                    -2.85386
Max. Drawdown Duration       45 days 01:28:00
Avg. Drawdown Duration        4 days 02:11:00
# Trades                          

/tmp/ipykernel_5993/3700659568.py:5: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run()


In [15]:
# optimizamos buscando mejores rendimientos
stats = bt.optimize(
    sl_pct=[0.005, 0.01, 0.02],
    tp_pct=[0.02, 0.03, 0.05],
    window=range(10, 50, 5),
    maximize='Sharpe Ratio'
)

print(stats)

/home/pulpo/miniconda3/envs/trading_env/lib/python3.11/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/pulpo/miniconda3/envs/trading_env/lib/python3.11/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/pulpo/miniconda3/envs/trading_env/lib/python3.11/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/pulpo/miniconda3/envs/trading_env/lib/python3.11/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. 

Start                     2026-03-06 14:32...
End                       2026-04-24 19:58...
Duration                     49 days 05:26:00
Exposure Time [%]                     88.0129
Equity Final [$]                  11545.83117
Equity Peak [$]                   11735.64251
Commissions [$]                     403.40771
Return [%]                           15.45831
Buy & Hold Return [%]                 5.90693
Return (Ann.) [%]                   192.01383
Volatility (Ann.) [%]                62.11767
CAGR [%]                            108.72162
Sharpe Ratio                          3.09113
Sortino Ratio                        18.70178
Calmar Ratio                           37.334
Alpha [%]                            13.02712
Beta                                  0.41158
Max. Drawdown [%]                    -5.14314
Avg. Drawdown [%]                    -0.40872
Max. Drawdown Duration       14 days 00:08:00
Avg. Drawdown Duration        0 days 12:09:00
# Trades                          

/home/pulpo/miniconda3/envs/trading_env/lib/python3.11/site-packages/backtesting/backtesting.py:1545: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = self.run(**dict(zip(heatmap.index.names, best_params)))


In [16]:
# grafica
bt.plot()

/home/pulpo/miniconda3/envs/trading_env/lib/python3.11/site-packages/bokeh/util/serialization.py:247: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))


GridPlot(id='p2485', ...)

In [18]:
# NUEVA DESCARGA: Datos que el bot NO conoce (Enero y Febrero)
df_test = yf.download('AAPL', start='2026-01-01', end='2026-03-01', interval='1h')

# Limpieza rápida como ya sabes hacer
df_test.columns = df_test.columns.get_level_values(0)
df_test.dropna(inplace=True)

# CLASE CON PARÁMETROS FIJOS (Los que ganaron en tu optimización)
class Robust_STD_Strategy(Strategy):
    # Aquí pon los valores exactos que te imprimió el "stats" de la optimización
    sl_pct = 0.02   # Ejemplo basado en tu tabla
    tp_pct = 0.05   
    window = 45     
    num_std = 2     

    def init(self):
        close = self.data.Close
        self.sma = self.I(lambda x: pd.Series(x).rolling(self.window).mean(), close)
        self.rolling_std = self.I(lambda x: pd.Series(x).rolling(self.window).std(), close)

    def next(self):
        price = self.data.Close[-1]
        media = self.sma[-1]
        desviacion = self.rolling_std[-1]

        if self.position:
            return
        
        if price < (media - self.num_std * desviacion):
            self.open_long(price)
        elif price > (media + self.num_std * desviacion):
            self.open_short(price)

    def open_long(self, price):
        self.buy(sl=price * (1 - self.sl_pct), tp=price * (1 + self.tp_pct))

    def open_short(self, price):
        self.sell(sl=price * (1 + self.sl_pct), tp=price * (1 - self.sl_pct))

# EJECUCIÓN SIMPLE (Sin optimizar)
bt_robust = Backtest(df_test, Robust_STD_Strategy, cash=10000, commission=0.002)
stats_robust = bt_robust.run()
print(stats_robust)
bt_robust.plot()

[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_5993/692896348.py:44: UserWarning: Superimposed OHLC plot matches the original plot. Skipping.
  bt_robust.plot()


Start                     2026-01-02 14:30...
End                       2026-02-27 20:30...
Duration                     56 days 06:00:00
Exposure Time [%]                    65.15152
Equity Final [$]                   9963.16926
Equity Peak [$]                   10185.58453
Commissions [$]                      428.5104
Return [%]                           -0.36831
Buy & Hold Return [%]                  1.3663
Return (Ann.) [%]                    -2.35603
Volatility (Ann.) [%]                22.08359
CAGR [%]                             -1.63947
Sharpe Ratio                         -0.10669
Sortino Ratio                        -0.14601
Calmar Ratio                         -0.40407
Alpha [%]                            -0.25007
Beta                                 -0.08654
Max. Drawdown [%]                    -5.83077
Avg. Drawdown [%]                    -3.76698
Max. Drawdown Duration       30 days 04:00:00
Avg. Drawdown Duration       13 days 19:00:00
# Trades                          

/home/pulpo/miniconda3/envs/trading_env/lib/python3.11/site-packages/bokeh/util/serialization.py:247: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))


GridPlot(id='p3212', ...)

In [19]:
stats_opt = bt_robust.optimize(
    sl_pct=[0.01, 0.02, 0.03],
    tp_pct=[0.03, 0.05, 0.08],
    window=range(20, 80, 10),
    num_std=[1.5, 2, 2.5],
    maximize='Sharpe Ratio'
)
print(stats_opt)

/home/pulpo/miniconda3/envs/trading_env/lib/python3.11/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/pulpo/miniconda3/envs/trading_env/lib/python3.11/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/pulpo/miniconda3/envs/trading_env/lib/python3.11/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/pulpo/miniconda3/envs/trading_env/lib/python3.11/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. 

Start                     2026-01-02 14:30...
End                       2026-02-27 20:30...
Duration                     56 days 06:00:00
Exposure Time [%]                    45.45455
Equity Final [$]                   10630.3663
Equity Peak [$]                    10630.3663
Commissions [$]                     199.51615
Return [%]                            6.30366
Buy & Hold Return [%]                 2.26068
Return (Ann.) [%]                    48.43708
Volatility (Ann.) [%]                24.29084
CAGR [%]                             31.50312
Sharpe Ratio                          1.99405
Sortino Ratio                         5.49314
Calmar Ratio                         12.31219
Alpha [%]                             6.30503
Beta                                 -0.00061
Max. Drawdown [%]                    -3.93408
Avg. Drawdown [%]                    -1.36518
Max. Drawdown Duration       14 days 20:00:00
Avg. Drawdown Duration        3 days 23:00:00
# Trades                          